# Laboratorio 04. Propagación hacia adelante

**Pregunta central:** ¿Cómo se componen las capas y cómo verificamos dimensiones y valores intermedios?

## Objetivos

- Seguir una entrada desde la primera capa hasta la salida.
- Inspeccionar dimensiones de activaciones.
- Comprobar que `nn.Linear` reproduce la operación matricial.

## Prerrequisitos y conexión

Retoma el MLP con NumPy del laboratorio 03. Se necesitan las dimensiones de matrices y la operación forward, pero todavía no se ajustan pesos.

## Teoría breve

Una capa lineal calcula $Z=XW+b$ y la activación transforma cada elemento. El orden de operaciones y las dimensiones determinan si la composición está bien definida.

## Problema y datos

Propagamos las cuatro entradas binarias de la tabla XOR con pesos pequeños fijados a mano; no descargamos datos ni entrenamos parámetros.

## Evaluación

La comprobación es la diferencia absoluta máxima entre NumPy y PyTorch, junto con las formas de todas las activaciones. Debe quedar dentro de tolerancia numérica.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Propagar con NumPy
Calculamos pesos, sesgos, activación ReLU y dimensiones intermedias.


In [ ]:
X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]], dtype=np.float32)
W1 = np.array([[1, -1], [-1, 1]], dtype=np.float32); b1 = np.array([0, 0], dtype=np.float32)
W2 = np.array([[1], [1]], dtype=np.float32); b2 = np.array([-0.5], dtype=np.float32)
z1 = X @ W1 + b1
h = np.maximum(z1, 0)
output = h @ W2 + b2
print('X:', X.shape, 'z1:', z1.shape, 'h:', h.shape, 'output:', output.shape)
print(output.ravel())


**Resultado e interpretación.** Cada matriz debe tener dimensiones compatibles: las unidades ocultas forman una representación intermedia; sus valores dependen de los parámetros iniciales.


### Reproducirlo con capas
Copiamos los parámetros a PyTorch y contrastamos cada salida.


In [ ]:
layer1, layer2 = nn.Linear(2, 2), nn.Linear(2, 1)
with torch.no_grad():
    layer1.weight.copy_(torch.tensor(W1.T)); layer1.bias.copy_(torch.tensor(b1))
    layer2.weight.copy_(torch.tensor(W2.T)); layer2.bias.copy_(torch.tensor(b2))
output_torch = layer2(torch.relu(layer1(torch.tensor(X))))
print('NumPy:', output.ravel().round(5))
print('PyTorch:', output_torch.detach().ravel().numpy().round(5))


**Resultado e interpretación.** La igualdad numérica verifica la propagación hacia adelante; todavía no hay aprendizaje porque los parámetros se copiaron a mano.


## Limitaciones y conclusiones

La equivalencia se verifica con una arquitectura pequeña y parámetros fijados a mano. Una salida idéntica comprueba la implementación del forward para este caso; no evalúa aprendizaje ni generalización.

## Ejercicios

1. **Guiado:** cambia los pesos de la capa final en NumPy y copia el mismo cambio a `nn.Linear`; compara las salidas.
2. **Independiente:** añade una tercera capa y anota las formas de cada matriz intermedia antes de ejecutar.
3. **Desafío:** implementa el forward de una red residual y explica cómo debe coincidir la forma de la entrada con la suma.

## Conexión con el siguiente laboratorio

Con el forward claro, el próximo paso propaga el error hacia atrás para calcular gradientes de los parámetros.